# 网络前向、反向与步长

核心梯度在 NumPy 中逐项可见，不用训练入口隐藏计算。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(20260907)
np.set_printoptions(precision=5, suppress=True)

标量两层 ReLU 网络的一次梯度。

In [ ]:
x,y=2.,1.;a,b,c,d=1.,-1.,2.,0.
z=a*x+b;hidden=max(z,0);prediction=c*hidden+d;error=prediction-y
grad=np.array([error*c*(z>0)*x,error*c*(z>0),error*hidden,error])
print('hidden, output, loss, gradients:',hidden,prediction,.5*error**2,grad)
for rate in [.1,.01]:
    aa,bb,cc,dd=np.array([a,b,c,d])-rate*grad;out=cc*max(aa*x+bb,0)+dd
    print(rate,out,.5*(out-y)**2)

小型 tanh 网络学习非线性函数，每轮显式反向传播。

In [ ]:
x=rng.uniform(-2,2,(200,1));y=np.sin(x)+.1*rng.normal(size=x.shape);train=np.arange(140);valid=np.arange(140,200)
W1=rng.normal(0,.5,(1,8));b1=np.zeros(8);W2=rng.normal(0,.2,(8,1));b2=np.zeros(1);curves=[]
for epoch in range(500):
    h=np.tanh(x[train]@W1+b1);pred=h@W2+b2;delta=(pred-y[train])/len(train)
    gW2=h.T@delta;gb2=delta.sum(axis=0);dh=(delta@W2.T)*(1-h*h);gW1=x[train].T@dh;gb1=dh.sum(axis=0)
    W1-=.05*gW1;b1-=.05*gb1;W2-=.05*gW2;b2-=.05*gb2
    val=np.tanh(x[valid]@W1+b1)@W2+b2
    curves.append([np.mean((pred-y[train])**2),np.mean((val-y[valid])**2)])
plt.plot(curves);plt.xlabel('epoch');plt.ylabel('MSE');plt.legend(['train','validation']);plt.show()

## 自己试一试

为什么学习率 0.1 的那一步损失上升？

## 反馈

它跨到 ReLU 零激活边界；负梯度只给足够小步长的局部下降。0.01 的更新在本例下降。

参数改变后应重新解释结果，不要求复现某次随机实验的小数。